# ERA5 wind data to CSV or Excel

Turns an ERA5 GRIB file into hourly tables of wind speed, wind direction, air temperature and air pressure at 100 m.

Install the packages once from a terminal:

```
pip install xarray cfgrib eccodes pandas numpy openpyxl jupyterlab
```

## Step 0: Get the data

Log in at https://cds.climate.copernicus.eu, then open https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download and select:

- Product type: Reanalysis
- Variable: 100m u-component of wind, 100m v-component of wind, 2m temperature, Surface pressure
- Year: one year
- Month, Day, Time: Select all
- Geographical area: Area selection. Select a small box around your site, 0.25° tall and wide (type the edges, or hold Shift and drag on the map), for example North 55.25, South 55, West 10, East 10.25
- Data format: GRIB
- Download format: Unarchived

Accept the licence, press "Submit form" and download the file from "Your requests" when it is ready (this takes about 30 minutes). Put the `.grib` file in the same folder as this notebook.

## Step 1: Load the packages

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

## Step 2: Settings

Paste the name of your GRIB file in `GRIB_FILE`. Choose `OUTPUT_TYPE`:

- `"one csv"`: one CSV file for the first grid point in the file (the North and West corner of your box)
- `"many csv"`: one CSV file for every grid point in the file
- `"excel"`: one Excel file with a sheet for every grid point

In [ ]:
GRIB_FILE = "adbd3f97b5a8942c8a8e371832c4902c.grib"    # replace with your own file's name
OUTPUT_FOLDER = "output"       # folder for the results, it is created if it does not exist
OUTPUT_TYPE = "one csv"        # "one csv", "many csv" or "excel"
SITE_TIMEZONE = 1              # time zone of the site in hours from UTC (Denmark is +1)

## Step 3: Open the GRIB file

In [ ]:
if not Path(GRIB_FILE).exists():
    raise FileNotFoundError(f"Cannot find {GRIB_FILE}. Check the name in step 2 and that the file is in the same folder as this notebook.")

if OUTPUT_TYPE not in ["excel", "one csv", "many csv"]:
    raise ValueError(f'OUTPUT_TYPE is "{OUTPUT_TYPE}", but it must be "excel", "one csv" or "many csv". Fix it in step 2.')

data = xr.open_dataset(GRIB_FILE, engine="cfgrib", backend_kwargs={"indexpath": ""})
data = data.astype("float64")   # ERA5 stores short numbers, without this Excel shows 17.95 as 17.95000076

for variable in ["u100", "v100", "t2m", "sp"]:
    if variable not in data:
        raise ValueError(f"The file has no {variable}. Download the data again with all four variables ticked (see step 0).")

print("The file is fine.")
print("First hour:", data["time"].values[0])
print("Last hour: ", data["time"].values[-1])
print("Number of hours:", len(data["time"]))

## Step 4: List the grid points

In [ ]:
latitudes = data["latitude"].values
longitudes = data["longitude"].values

print("Number of grid points in the file:", len(latitudes) * len(longitudes))
number = 0
for latitude in latitudes:
    for longitude in longitudes:
        number = number + 1
        print(f"  point {number}: {latitude} N, {longitude} E")

## Step 5: Calculations

In [ ]:
def make_table(latitude, longitude):
    """Compute the four columns for one grid point and return them as a table."""
    point = data.sel(latitude=latitude, longitude=longitude)

    u = point["u100"].values             # m/s
    v = point["v100"].values             # m/s
    temperature_2m = point["t2m"].values # kelvin
    pressure_ground = point["sp"].values # pascal

    wind_speed = np.sqrt(u**2 + v**2)
    wind_direction = np.degrees(np.arctan2(-u, -v))
    wind_direction = wind_direction % 360    # turns for example -90 into 270

    temperature_100m = temperature_2m - 0.0065 * (100 - 2)
    pressure_100m = pressure_ground * np.exp(-9.81 * 100 / (287.05 * temperature_100m))

    table = pd.DataFrame()
    table["wind speed at 100m (m/s)"] = wind_speed.round(2)
    table["wind direction at 100m (deg)"] = wind_direction.round(2)
    table["air temperature at 100m (C)"] = (temperature_100m - 273.15).round(2)
    table["air pressure at 100m (Pa)"] = pressure_100m.round(0).astype(int)
    return table


def make_header(site_id, latitude, longitude):
    """The first line of the output, with information about the site.

    ERA5 times are in UTC, so the data time zone is 0.
    """
    return [
        "SiteID", site_id,
        "Site Timezone", SITE_TIMEZONE,
        "Data Timezone", 0,
        "Longitude", float(longitude),
        "Latitude", float(latitude),
    ]


def save_csv(table, header, file_path):
    """Save one table as a CSV file: the header line first, then the table."""
    with open(file_path, "w", newline="") as file:
        writer = csv.writer(file, lineterminator="\r\n")
        writer.writerow(header)
        table.to_csv(file, index=False, lineterminator="\r\n")
    print("Saved", file_path)

## Step 6: Save the results

The first row is the first hour in the GRIB file, in UTC.

In [ ]:
output_folder = Path(OUTPUT_FOLDER)
output_folder.mkdir(exist_ok=True)
name = Path(GRIB_FILE).stem    # the GRIB file name without .grib


if OUTPUT_TYPE == "one csv":
    latitude = latitudes[0]
    longitude = longitudes[0]
    print(f"Using the first grid point in the file: {latitude} N, {longitude} E")

    table = make_table(latitude, longitude)
    header = make_header(1, latitude, longitude)
    save_csv(table, header, output_folder / f"{name}_{latitude}N_{longitude}E.csv")


if OUTPUT_TYPE == "many csv":
    number = 0
    for latitude in latitudes:
        for longitude in longitudes:
            number = number + 1
            table = make_table(latitude, longitude)
            header = make_header(number, latitude, longitude)
            save_csv(table, header, output_folder / f"{name}_{latitude}N_{longitude}E.csv")


if OUTPUT_TYPE == "excel":
    excel_path = output_folder / f"{name}.xlsx"
    with pd.ExcelWriter(excel_path, engine="openpyxl") as excel_file:
        number = 0
        for latitude in latitudes:
            for longitude in longitudes:
                number = number + 1
                sheet_name = f"{latitude}N_{longitude}E"

                # The table starts on row 2, so row 1 is free for the header line
                table = make_table(latitude, longitude)
                table.to_excel(excel_file, sheet_name=sheet_name, startrow=1, index=False)

                header = make_header(number, latitude, longitude)
                sheet = excel_file.sheets[sheet_name]
                for column in range(len(header)):
                    sheet.cell(row=1, column=column + 1, value=header[column])   # Excel counts columns from 1

                print("Added sheet", sheet_name)
    print("Saved", excel_path)

## Step 7: Check the result

In [ ]:
print(len(table), "hours")
table.describe()